# Tech Challenge - Fase 3: Predição e Inteligência Analítica para Alfabetização no Brasil
**Pós Tech em Data Science & Machine Learning**

Este notebook apresenta o desenvolvimento de ponta a ponta do modelo de classificação supervisionada para prever a condição de alfabetização infantil de alunos no 2º ano do Ensino Fundamental com base em variáveis educacionais, territoriais e socioeconômicas.

In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Adicionar raiz ao PYTHONPATH
ROOT_DIR = Path('..').resolve()
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

from src.config import RANDOM_STATE, FIGURES_DIR, REPORTS_DIR
from src.data_loader import load_gold_silver_data
from src.eda import perform_eda
from src.preprocessing import split_and_preprocess_data
from src.models import get_candidate_models, evaluate_models_cross_validation, fit_and_save_all_models
from src.tuning import tune_lightgbm_optuna
from src.evaluation import evaluate_all_models_on_test, optimize_decision_threshold
from src.explainability import explain_model_with_shap
print('Ambiente configurado com sucesso!')

Ambiente configurado com sucesso!


## 1. Carga e Integração dos Dados (Camadas Silver/Gold + Microdados)
Consolidação das informações de alunos, escolas, municípios e dados socioeconômicos.

In [2]:
df_raw = load_gold_silver_data(sample_size=30000, seed=RANDOM_STATE)
print('Dimensões do DataFrame:', df_raw.shape)
df_raw.head()

[DATA LOADER] Lendo dados reais de alunos da camada Silver: C:\Users\Pedro.Cardoso\Downloads\gold-main\data\silver\fato_aluno_alfabetizacao\execution_date=2026-08-31\ano=2024\fato_aluno_alfabetizacao.parquet
[DATA LOADER] Amostra consolidada de alunos: 30,000 registros.
[DATA LOADER] Dataset final pronto com 30,000 linhas e 23 colunas.
Dimensões do DataFrame: (30000, 23)


,alfabetizado,rede,sigla_uf,regiao_brasil,localizacao,porte_municipio,frequencia_escolar,formacao_docente_superior,tamanho_turma,horas_aula_diarias,...,infra_biblioteca,infra_laboratorio_info,infra_internet_banda_larga,infra_quadra_esportes,beneficiario_bolsa_familia,renda_per_capita_reais,escolaridade_mae,tem_computador_ou_tablet,acesso_internet_casa,quantidade_livros_casa
0,0,Estadual,DF,Centro-Oeste,Urbana,Medio,69.7,86.7,28,4.5,...,Não,Não,Sim,Sim,Sim,520.65,Sem instrucao,Sim,Sim,4
1,0,Municipal,RJ,Sudeste,Rural,Pequeno II,72.0,83.4,15,4.0,...,Não,Não,Não,Não,Não,1282.03,Medio completo,Não,Sim,13
2,1,Municipal,SP,Sudeste,Urbana,Metropole,97.3,95.3,25,5.0,...,Sim,Não,Não,Sim,Não,1759.26,Fundamental completo,Sim,Sim,8
3,1,Municipal,MT,Centro-Oeste,Urbana,Pequeno I,73.1,79.4,25,4.5,...,Não,Não,Sim,Não,Sim,925.55,Medio completo,Não,Sim,25
4,1,Estadual,SP,Sudeste,Urbana,Medio,86.2,79.0,20,4.5,...,Sim,Sim,Sim,Sim,Sim,476.41,Superior completo,Não,Sim,13


## 2. Análise Exploratória dos Dados (EDA)
Inspeção de distribuições, missing values, outliers e análise de balanceamento da classe alvo.

In [3]:
eda_results = perform_eda(df_raw, save_figures=True)

 1. ANÁLISE EXPLORATÓRIA DOS DADOS (EDA) & DIAGNÓSTICO ESTATÍSTICO

[EDA] Dimensões do Dataset: 30,000 linhas e 23 colunas.

--- Diagnóstico de Valores Ausentes ---
                           Total Ausentes  Percentual (%)
renda_per_capita_reais                925            3.08
frequencia_escolar                    717            2.39
formacao_docente_superior             621            2.07
escolaridade_mae                      452            1.51

--- Distribuição da Variável Alvo (Alfabetização) ---
Classe 1 (Alfabetizado):     15,663 (52.21%)
Classe 0 (Não Alfabetizado): 14,337 (47.79%)
Razão de Desbalanceamento:  1.09:1

--- Detecção de Outliers via Intervalo Interquartil (IQR) ---
  - frequencia_escolar        :    48 outliers (0.16%) | Limites: [48.9, 113.9]
  - formacao_docente_superior :    72 outliers (0.25%) | Limites: [58.2, 103.8]
  - horas_aula_diarias        : 2,403 outliers (8.01%) | Limites: [3.2, 5.2]
  - ivs_territorial           :     8 outliers (0.03%) | Limites:

## 3. Engenharia de Atributos e Pré-processamento (Zero Data Leakage)
Criação dos índices compostos e pipelines de transformação ajustados exclusivamente no conjunto de treino.

In [4]:
X_train, X_test, y_train, y_test, feature_names, preprocessor = split_and_preprocess_data(df_raw)
print('Total de features geradas:', len(feature_names))


 2. ENGENHARIA DE ATRIBUTOS E PRÉ-PROCESSAMENTO (ZERO DATA LEAKAGE)
[PREPROCESSING] Divisão Estratificada Concluída:
  - Conjunto de Treino: 24,000 amostras (80%)
  - Conjunto de Teste:  6,000 amostras (20%)
  - Proporção Classe 1 (Treino): 52.21% | (Teste): 52.22%
[PREPROCESSING] Ajustando pipeline de transformadores no Treino (fit_transform)...
[PREPROCESSING] Aplicando transformadores no Teste (transform)...
[PREPROCESSING] Total de features geradas após codificação: 54
Total de features geradas: 54


## 4. Modelagem & Validação Cruzada Estratificada (5-Fold CV)
Avaliação comparativa: Baseline (Regressão Logística) vs Random Forest vs XGBoost vs LightGBM.

In [5]:
candidate_models = get_candidate_models()
cv_comparison = evaluate_models_cross_validation(candidate_models, X_train, y_train)
trained_models = fit_and_save_all_models(candidate_models, X_train, y_train)


 3. MODELAGEM & VALIDAÇÃO CRUZADA ESTRATIFICADA (5-FOLD CV)
[MODELING] Treinando e validando via CV: Baseline_Logistic_Regression   ... [OK]
[MODELING] Treinando e validando via CV: Random_Forest                  ... [OK]
[MODELING] Treinando e validando via CV: XGBoost                        ... [OK]
[MODELING] Treinando e validando via CV: LightGBM                       ... [OK]

--- Resultados da Validação Cruzada (Média ± Desvio Padrão) ---
                      Modelo         ROC-AUC          PR-AUC        F1-Score          Recall       Precision        Accuracy
Baseline_Logistic_Regression 0.8698 (±0.004) 0.8769 (±0.006) 0.7917 (±0.003) 0.7837 (±0.008) 0.8000 (±0.009) 0.7847 (±0.004)
                     XGBoost 0.8671 (±0.005) 0.8746 (±0.006) 0.7931 (±0.003) 0.7970 (±0.007) 0.7894 (±0.009) 0.7829 (±0.004)
                    LightGBM 0.8662 (±0.005) 0.8727 (±0.006) 0.7878 (±0.006) 0.7778 (±0.010) 0.7983 (±0.011) 0.7812 (±0.006)
               Random_Forest 0.8633 (±0.005) 0.867

## 5. Otimização Bayesiana de Hiperparâmetros (Optuna)
Ajuste fino dos hiperparâmetros do modelo LightGBM.

In [6]:
best_lgbm_model, best_params, best_cv_score = tune_lightgbm_optuna(X_train, y_train, n_trials=25)
trained_models['LightGBM_Optimized'] = best_lgbm_model


 3.1 OTIMIZAÇÃO DE HIPERPARÂMETROS VIA OPTUNA (BAYESIAN SEARCH)
[TUNING] Iniciando busca bayesiana (25 trials) com Stratified 5-Fold CV...
[TUNING] Otimização Concluída!
  - Melhor ROC-AUC CV: 0.8680
  - Hiperparâmetros Ótimos:
      * n_estimators        : 150
      * max_depth           : 4
      * num_leaves          : 48
      * learning_rate       : 0.04401
      * subsample           : 0.73051
      * colsample_bytree    : 0.82379
      * min_child_samples   : 16
      * reg_alpha           : 1.45150
      * reg_lambda          : 0.00794
[TUNING] Modelo otimizado salvo em: C:\Users\Pedro.Cardoso\Downloads\gold-main\models_saved\lightgbm_optimized.joblib


## 6. Avaliação de Desempenho no Teste Independente e Calibração de Limiar
Avaliação das métricas (ROC-AUC, PR-AUC, F1, Recall, Precision) e calibração de corte social.

In [7]:
test_metrics_df = evaluate_all_models_on_test(trained_models, X_test, y_test, save_figures=True)
threshold_results = optimize_decision_threshold(best_lgbm_model, X_test, y_test, save_figures=True)


 4. AVALIAÇÃO DE DESEMPENHO NO CONJUNTO DE TESTE INDEPENDENTE

--- Tabela Comparativa no Conjunto de Teste (20% Holdout) ---
                      Modelo  Acurácia  ROC-AUC  PR-AUC  F1-Score  Recall (Alfabetizado)  Recall (Não Alfab - Crítico)  Precisão  Log-Loss  Brier Score
Baseline_Logistic_Regression    0.7943   0.8805  0.8859    0.8014                 0.7948                        0.7939    0.8082    0.4352       0.1403
          LightGBM_Optimized    0.7910   0.8778  0.8842    0.7977                 0.7890                        0.7932    0.8065    0.4398       0.1421
                     XGBoost    0.7923   0.8767  0.8831    0.8022                 0.8063                        0.7771    0.7981    0.4400       0.1422
                    LightGBM    0.7910   0.8762  0.8823    0.7979                 0.7900                        0.7921    0.8059    0.4415       0.1428
               Random_Forest    0.7927   0.8734  0.8760    0.8011                 0.7996                        0.

## 7. Explicabilidade do Modelo (SHAP - Explainable AI)
Interpretação dos fatores determinantes da alfabetização.

In [8]:
shap_results = explain_model_with_shap(best_lgbm_model, X_test, feature_names, sample_size=1500, save_figures=True)


 5. EXPLICABILIDADE DO MODELO (EXPLAINABLE AI - SHAP)
[SHAP] Calculando valores SHAP via TreeExplainer (1,500 amostras)...

--- Top 10 Variáveis Mais Determinantes (Global Feature Importance SHAP) ---
   1. frequencia_escolar                  | Educacional     | Impacto Médio: 1.3317
   2. razao_engajamento_turma             | Educacional     | Impacto Médio: 0.1140
   3. sigla_uf_CE                         | Territorial     | Impacto Médio: 0.1018
   4. tamanho_turma                       | Educacional     | Impacto Médio: 0.0714
   5. sigla_uf_MG                         | Territorial     | Impacto Médio: 0.0514
   6. sigla_uf_BA                         | Territorial     | Impacto Médio: 0.0437
   7. sigla_uf_PR                         | Territorial     | Impacto Médio: 0.0348
   8. regiao_brasil_Norte                 | Territorial     | Impacto Médio: 0.0346
   9. sigla_uf_GO                         | Territorial     | Impacto Médio: 0.0321
  10. sigla_uf_RS                         

C:\Users\Pedro.Cardoso\Downloads\gold-main\src\explainability.py:117: FutureWarning: The NumPy global RNG was seeded by calling `np.random.seed`. In a future version this function will no longer use the global RNG. Pass `rng` explicitly to opt-in to the new behaviour and silence this warning.
  shap.summary_plot(shap_values, df_sample, max_display=15, show=False)


[SHAP] Gráficos de interpretabilidade salvos em: C:\Users\Pedro.Cardoso\Downloads\gold-main\reports\figures
